
# 01 – Download Olist dataset to landing zone

**Purpose:** Downloads the public [Olist Brazilian E-Commerce dataset](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) from Kaggle and places each CSV file into its own folder in the Unity Catalog landing volume.

**Why:** The landing volume simulates a file drop from a source system (e.g. SFTP export). Each table gets a dedicated folder so that Auto Loader can ingest it independently into the bronze layer.

| | |
|---|---|
| **Source** | Kaggle – `olistbr/brazilian-ecommerce` (public, no API token required) |
| **Target** | `/Volumes/<catalog>/landing/raw/olist/<table>/` |
| **Run** | Once, during environment setup (after `00_unity_catalog.sql`) |
| **Idempotent** | Yes – re-running overwrites files with the same content |

**Data license:** CC BY-NC-SA 4.0 – raw data is not committed to this repository.

In [0]:
# Install Kaggle client (only needed for this setup notebook)
%pip install kagglehub

## Configuration
Catalog is a parameter so the same notebook works for `dev` and `prod` environments.

In [0]:
import os
import shutil

import kagglehub

# Catalog as a widget parameter -> environment-agnostic notebook
dbutils.widgets.text("catalog", "olist_dev", "Target catalog")
CATALOG = dbutils.widgets.get("catalog")

KAGGLE_DATASET = "olistbr/brazilian-ecommerce"
TARGET_ROOT = f"/Volumes/{CATALOG}/landing/raw/olist"

# Mapping: target folder (= future bronze table name) -> source file name
FILES: dict[str, str] = {
    "customers":            "olist_customers_dataset.csv",
    "geolocation":          "olist_geolocation_dataset.csv",
    "orders":               "olist_orders_dataset.csv",
    "order_items":          "olist_order_items_dataset.csv",
    "order_payments":       "olist_order_payments_dataset.csv",
    "order_reviews":        "olist_order_reviews_dataset.csv",
    "products":             "olist_products_dataset.csv",
    "sellers":              "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

print(f"Target: {TARGET_ROOT}")

## Download
`kagglehub` downloads the dataset into a temporary local cache on the compute node.
The cache is ephemeral, so files must be copied to the Unity Catalog volume in the next step.

In [0]:
source_dir = kagglehub.dataset_download(KAGGLE_DATASET)
print(f"Downloaded to local cache: {source_dir}")

## Copy to landing volume
Each file is copied into its own folder. The run fails fast if any expected file is missing,
so an incomplete dataset never reaches the bronze layer.

In [0]:
def copy_to_landing(source_dir: str, target_root: str, files: dict[str, str]) -> None:
    """Copy each source file into its own table folder in the landing volume."""
    missing = [f for f in files.values() if not os.path.exists(os.path.join(source_dir, f))]
    if missing:
        raise FileNotFoundError(f"Missing files in downloaded dataset: {missing}")

    for folder, file_name in files.items():
        target_dir = os.path.join(target_root, folder)
        os.makedirs(target_dir, exist_ok=True)
        shutil.copy(os.path.join(source_dir, file_name), os.path.join(target_dir, file_name))
        print(f"OK  {file_name:<42} -> {folder}/")


copy_to_landing(source_dir, TARGET_ROOT, FILES)

## Validate
Row count per table as a quick sanity check (expected ~99,441 rows in `orders`).

In [0]:
results = []
for folder in FILES:
    df = spark.read.csv(f"{TARGET_ROOT}/{folder}/", header=True)
    results.append((folder, df.count()))

display(spark.createDataFrame(results, ["table", "row_count"]))